In [1]:
!pip install ucimlrepo lightgbm onnxmltools skl2onnx onnxruntime imbalanced-learn pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.0/304.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.1 MB/s eta 0:00:00:00:0100:01


In [4]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

# ONNX Conversion dependencies
from skl2onnx import convert_sklearn, update_registered_converter
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from onnxmltools.convert.lightgbm.operator_converters.LightGbm import convert_lightgbm
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

print("Step 1: Programmatically fetching CDC Diabetes Health Indicators dataset...")
dataset = fetch_ucirepo(id=891) 
X = dataset.data.features
y = dataset.data.targets.iloc[:, 0].values.astype(int)

print(f"Loaded {X.shape[0]} records with {X.shape[1]} clinical/lifestyle features.")

print("\nStep 2: Splitting dataset into Train (60%), Validation (12%), and Test (28%) splits...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.70, random_state=42, stratify=y_temp
)

print(f"-> Train Set: {X_train.shape[0]} samples")
print(f"-> Validation Set: {X_val.shape[0]} samples")
print(f"-> Test Set: {X_test.shape[0]} samples")

print("\nStep 3: Correcting class imbalance via SMOTE on the training set...")
smote = SMOTE(random_state=42)
X_train_res_array, y_train_res = smote.fit_resample(X_train, y_train)

# Convert the resampled numpy array back to a Pandas DataFrame to preserve feature names and suppress warnings
X_train_res = pd.DataFrame(X_train_res_array, columns=X_train.columns)
print(f"Balanced training labels: {np.bincount(y_train_res)}")

print("\nStep 4: Designing and training the Scikit-learn Machine Learning pipeline...")
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lgbm', lgb.LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        random_state=42,
        objective='binary',
        verbose=-1
    ))
])

pipe.fit(X_train_res, y_train_res)

# Evaluate model performance against the holdout test set
y_pred = pipe.predict(X_test)
print("\n--- Scikit-Learn Model Performance (Test Set) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

print("\nStep 5: Converting the trained pipeline into ONNX format...")
# Register the LightGBM converter with the skl2onnx registry
update_registered_converter(
    lgb.LGBMClassifier, 
    'LightGbmLGBMClassifier',
    calculate_linear_classifier_output_shapes, 
    convert_lightgbm,
    options={'nocl': [False], 'zipmap': [True, False, "columns"]}
)

# Define input tensor format with dynamic batch sizes (None)
initial_type = [('input', FloatTensorType([None, X.shape[1]]))]

# Target explicit domain versions to resolve the version incompatibility error
target_opset_dict = {
    '': 12,              # Standard ONNX operators
    'ai.onnx.ml': 3      # Machine Learning extension operators for LightGBM/Trees
}

onnx_model = convert_sklearn(
    pipe, 
    name='HealthAIoT_Diabetes_Predictor',
    initial_types=initial_type,
    target_opset=target_opset_dict,
    options={'zipmap': False}  # Exclude zipmap to keep web-runtime parsing simple
)

# Export model
output_filename = "diabetes_predictor.onnx"
with open(output_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())
print(f"Model saved as '{output_filename}'")

print("\nStep 6: Verifying ONNX mathematical parity...")

# 1. Suppress ONNX Runtime's C++ warning logs by setting log_severity_level to 3 (Error/Fatal only)
sess_opts = ort.SessionOptions()
sess_opts.log_severity_level = 3  
session = ort.InferenceSession(output_filename, sess_options=sess_opts)

input_name = session.get_inputs()[0].name

# 2. Maintain a Pandas DataFrame slice for Scikit-Learn to prevent Feature Name warnings
sample_inputs_df = X_test[:5].astype(np.float32)

# 3. Create a raw NumPy array ONLY for ONNX Runtime
sample_inputs_onnx = sample_inputs_df.values

# Execute Python-based calculations (using the DataFrame)
sklearn_probs = pipe.predict_proba(sample_inputs_df)

# Execute ONNX Runtime calculations (using the NumPy array)
onnx_outputs = session.run(None, {input_name: sample_inputs_onnx})
onnx_probs = onnx_outputs[1]

print("\nVerification Data (Probabilities of class 1):")
print(f"-> Scikit-Learn Pipeline:  {sklearn_probs[:, 1]}")
print(f"-> ONNX Runtime:           {onnx_probs[:, 1]}")

# Assert exact numerical parity
assert np.allclose(sklearn_probs, onnx_probs, atol=1e-5), "Discrepancy detected between Python and ONNX results."
print("\nParity verification complete. Predictions match to 5 decimal places with zero warnings.")

Step 1: Programmatically fetching CDC Diabetes Health Indicators dataset...
Loaded 253680 records with 21 clinical/lifestyle features.

Step 2: Splitting dataset into Train (60%), Validation (12%), and Test (28%) splits...
-> Train Set: 152208 samples
-> Validation Set: 30441 samples
-> Test Set: 71031 samples

Step 3: Correcting class imbalance via SMOTE on the training set...
Balanced training labels: [131000 131000]

Step 4: Designing and training the Scikit-learn Machine Learning pipeline...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Scikit-Learn Model Performance (Test Set) ---
Accuracy: 0.7179
F1-Score: 0.4100
              precision    recall  f1-score   support

           0       0.94      0.72      0.81     61134
           1       0.29      0.70      0.41      9897

    accuracy                           0.72     71031
   macro avg       0.61      0.71      0.61     71031
weighted avg       0.85      0.72      0.76     71031


Step 5: Converting the trained pipeline into ONNX format...
Model saved as 'diabetes_predictor.onnx'

Step 6: Verifying ONNX mathematical parity...

Verification Data (Probabilities of class 1):
-> Scikit-Learn Pipeline:  [0.41545438 0.82050175 0.56047744 0.01374186 0.79473672]
-> ONNX Runtime:           [0.4154544  0.8205017  0.56047744 0.01374185 0.7947367 ]

Parity verification complete. Predictions match to 5 decimal places with zero warnings.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
!pip install supabase sentence-transformers torch

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
from supabase import create_client, Client

# 1. Initialize connection to Supabase using your project credentials
SUPABASE_URL = "YOUR_SUPABASE_URL"
SUPABASE_ANON_KEY = "YOUR_SUPABASE_ANON_KEY"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

print("Step 1: Downloading open-source embedding model locally (all-MiniLM-L6-v2)...")
# This model outputs exactly 384 dimensions, matching our public.clinical_knowledge database schema
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded successfully.")

print("\nStep 2: Preparing clinical knowledge guidelines...")
clinical_guidelines = [
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "HighBP",
        "clinical_recommendation": "For people with diabetes and hypertension, the ADA recommends a target blood pressure of <130/80 mmHg. Lifestyle therapies include sodium restriction (<2,300 mg/day), weight loss, a DASH-style eating pattern, and increased physical activity."
    },
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "BMI",
        "clinical_recommendation": "For overweight or obese adults with type 2 diabetes risk, weight loss of at least 5-7% of initial body weight is highly recommended. This should be combined with intensive lifestyle interventions such as a healthy diet and weekly physical activity."
    },
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "HighChol",
        "clinical_recommendation": "Statin therapy is recommended for primary prevention of cardiovascular disease in adults with diabetes aged 40-75 years. Target LDL cholesterol reduction of 50% or more from baseline using moderate- to high-intensity statins."
    },
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "PhysActivity",
        "clinical_recommendation": "Adults with diabetes or prediabetes should engage in 150 minutes or more of moderate-to-vigorous intensity aerobic physical activity per week, spread over at least 3 days per week, with no more than 2 consecutive days without activity."
    },
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "Smoker",
        "clinical_recommendation": "All patients are advised not to use cigarettes, other tobacco products, or e-cigarettes. Incorporate smoking cessation counseling and pharmacological treatment options as standard preventive care."
    },
    {
        "source_citation": "ADA Standards of Care in Diabetes 2024",
        "risk_factor": "GenHlth",
        "clinical_recommendation": "Annual comprehensive assessments of microvascular and macrovascular complications are recommended. Self-monitored blood glucose tracking and self-reported health ratings help clinicians tailor pharmacotherapy."
    }
]

print("\nStep 3: Generating vector embeddings and inserting data into Supabase...")
for item in clinical_guidelines:
    text_to_embed = item["clinical_recommendation"]
    
    # Generate embedding and convert output numpy array to a python list of floats
    embedding_vector = model.encode(text_to_embed).tolist()
    
    # Map to schema payload
    payload = {
        "source_citation": item["source_citation"],
        "risk_factor": item["risk_factor"],
        "clinical_recommendation": item["clinical_recommendation"],
        "embedding": embedding_vector
    }
    
    # Insert data via Supabase PostgREST Client
    response = supabase.table("clinical_knowledge").insert(payload).execute()
    print(f"-> Successfully inserted guideline for risk factor: {item['risk_factor']}")

print("\nAll clinical guidelines have been embedded and securely uploaded to Supabase.")

Step 1: Downloading open-source embedding model locally (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.

Step 2: Preparing clinical knowledge guidelines...

Step 3: Generating vector embeddings and inserting data into Supabase...
-> Successfully inserted guideline for risk factor: HighBP
-> Successfully inserted guideline for risk factor: BMI
-> Successfully inserted guideline for risk factor: HighChol
-> Successfully inserted guideline for risk factor: PhysActivity
-> Successfully inserted guideline for risk factor: Smoker
-> Successfully inserted guideline for risk factor: GenHlth

All clinical guidelines have been embedded and securely uploaded to Supabase.


In [12]:
!pip install google-genai

In [13]:
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer
from supabase import create_client, Client
import json

# 1. Configuration
SUPABASE_URL = "YOUR_SUPABASE_URL"
SUPABASE_ANON_KEY = "YOUR_SUPABASE_ANON_KEY"
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"     # <-- Paste your free Gemini API key

# Initialize clients
supabase: Client = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

# Modern google-genai client initialization
client = genai.Client(api_key=GEMINI_API_KEY)

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Step 1: Simulating user incoming prediction payload...")
user_payload = {
    "risk_score": 0.8205,           # 82.05% prediction score from ONNX Runtime
    "features": {
        "HighBP": 1,                # User has High Blood Pressure
        "HighChol": 1,              # User has High Cholesterol
        "BMI": 28.0,                # Overweight range
        "PhysActivity": 0,          # Inactive lifestyle
        "Smoker": 0,
        "GenHlth": 4,               # Rated fair/poor
        "Age": 9                    # Standard CDC age category
    }
}

print("\nStep 2: Performing Semantic RAG Retrieval...")
high_risks = []
if user_payload["features"]["HighBP"] == 1: high_risks.append("high blood pressure")
if user_payload["features"]["BMI"] >= 25.0: high_risks.append("overweight body mass index BMI")
if user_payload["features"]["HighChol"] == 1: high_risks.append("high cholesterol")
if user_payload["features"]["PhysActivity"] == 0: high_risks.append("low physical activity sedentary lifestyle")

query_text = "preventive advice for " + " and ".join(high_risks)
print(f"-> Generated search query: '{query_text}'")

# Embed the search query
query_embedding = embedding_model.encode(query_text).tolist()

# Query Supabase using match_clinical_knowledge RPC function
rpc_response = supabase.rpc(
    "match_clinical_knowledge", 
    {
        "query_embedding": query_embedding,
        "match_threshold": 0.1, 
        "match_count": 2        
    }
).execute()

retrieved_guidelines = rpc_response.data
print(f"-> Retrieved {len(retrieved_guidelines)} matching clinical guidelines from Supabase.")

print("\nStep 3: Setting up the Compliant AI Clinical Agent...")
system_instruction = """
You are a highly secure, professional clinical interpreter for a smart healthcare system. 
Your task is to explain a patient's numerical diabetes risk metrics using only authorized clinical guidelines.

Strict Compliance Rules:
1. Do NOT diagnose the patient. Use words like 'Your risk profile suggests' or 'Guidelines advise'.
2. You must rely ONLY on the provided Clinical Context below. Do not assume or introduce medical knowledge outside this context.
3. Explicitly cite your clinical sources (e.g., 'According to the ADA Standards of Care 2024...').
4. Keep the explanation structured, patient-friendly, and actionable, emphasizing lifestyle modifications.
"""

# Format the context blocks for the prompt
context_block = ""
for idx, item in enumerate(retrieved_guidelines):
    context_block += f"\nSource [{idx+1}]: {item['source_citation']}\n"
    context_block += f"Category: {item['risk_factor']}\n"
    context_block += f"Recommendation: {item['clinical_recommendation']}\n"

prompt = f"""
Patient Risk Profile:
- Calculated Diabetes Risk Score: {user_payload['risk_score'] * 100:.2f}%
- Key Features: BMI: {user_payload['features']['BMI']}, High Blood Pressure: {'Yes' if user_payload['features']['HighBP'] == 1 else 'No'}, High Cholesterol: {'Yes' if user_payload['features']['HighChol'] == 1 else 'No'}, Physical Activity: {'Active' if user_payload['features']['PhysActivity'] == 1 else 'Inactive'}

Clinical Context Guidelines:
{context_block}

Based on the profile and context, write a safe, compliant clinical explanation and preventive action plan for the patient.
"""

print("\nStep 4: Executing LLM synthesis via the modern Gemini API...")
# Generate content using the new SDK syntax and the active gemini-2.5-flash model
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.2  # Low temperature guarantees deterministic, guideline-focused outputs
    )
)

print("\n--- Final Patient Explanation Output ---")
print(response.text)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step 1: Simulating user incoming prediction payload...

Step 2: Performing Semantic RAG Retrieval...
-> Generated search query: 'preventive advice for high blood pressure and overweight body mass index BMI and high cholesterol and low physical activity sedentary lifestyle'
-> Retrieved 2 matching clinical guidelines from Supabase.

Step 3: Setting up the Compliant AI Clinical Agent...

Step 4: Executing LLM synthesis via the modern Gemini API...

--- Final Patient Explanation Output ---
Hello. Based on the information from your health profile, I will explain your diabetes risk metrics and outline some preventive steps, strictly following authorized clinical guidelines.

Your calculated diabetes risk score of **82.05%** suggests a significant risk for developing type 2 diabetes. This score is influenced by several factors identified in your profile, including your BMI, current blood pressure status, cholesterol levels, and physical activity level.

Here are some key areas and recommende

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>HealthAIoT - Patient Dashboard</title>
    <!-- Tailwind CSS CDN for modern styling -->
    <script src="https://cdn.tailwindcss.com"></script>
    <!-- ONNX Runtime Web CDN to run the model locally in browser -->
    <script src="https://cdn.jsdelivr.net/npm/onnxruntime-web/dist/ort.min.js"></script>
</head>
<body class="bg-gray-50 min-h-screen text-gray-900">

    <header class="bg-indigo-600 text-white shadow-md py-4">
        <div class="container mx-auto px-4 flex justify-between items-center">
            <h1 class="text-2xl font-bold tracking-tight">HealthAIoT</h1>
            <span class="text-sm bg-indigo-500 px-3 py-1 rounded-full font-semibold">Local TinyML Mode Active</span>
        </div>
    </header>

    <main class="container mx-auto px-4 py-8 grid grid-cols-1 lg:grid-cols-2 gap-8">
        <!-- Left Column: Input Form -->
        <section class="bg-white p-6 rounded-2xl shadow-sm border border-gray-100">
            <h2 class="text-xl font-bold mb-4 text-indigo-700">Health & Lifestyle Questionnaire</h2>
            <form id="healthForm" class="space-y-4">
                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">High Blood Pressure?</label>
                        <select id="HighBP" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">High Cholesterol?</label>
                        <select id="HighChol" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Cholesterol Checked (Last 5 Yrs)?</label>
                        <select id="CholCheck" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">Yes</option>
                            <option value="0">No</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Body Mass Index (BMI)</label>
                        <input type="number" id="BMI" value="28.0" step="0.1" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-3 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Smoked >100 Cigarettes?</label>
                        <select id="Smoker" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">History of Stroke?</label>
                        <select id="Stroke" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Heart Disease/Attack?</label>
                        <select id="HeartDiseaseorAttack" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Physical Activity (Last 30 Days)?</label>
                        <select id="PhysActivity" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">Yes</option>
                            <option value="0">No</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Consume Fruits Daily?</label>
                        <select id="Fruits" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">Yes</option>
                            <option value="0">No</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Consume Veggies Daily?</label>
                        <select id="Veggies" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">Yes</option>
                            <option value="0">No</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Heavy Alcohol Consumer?</label>
                        <select id="HvyAlcoholConsump" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Any Healthcare Plan?</label>
                        <select id="AnyHealthcare" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">Yes</option>
                            <option value="0">No</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Could Not Afford Doctor?</label>
                        <select id="NoDocbcCost" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-3 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">General Health (1-5)</label>
                        <select id="GenHlth" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="1">1 (Excellent)</option>
                            <option value="2">2 (Very Good)</option>
                            <option value="3">3 (Good)</option>
                            <option value="4" selected>4 (Fair)</option>
                            <option value="5">5 (Poor)</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Mental Health (0-30 days)</label>
                        <input type="number" id="MentHlth" value="0" min="0" max="30" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Physical Health (0-30 days)</label>
                        <input type="number" id="PhysHlth" value="0" min="0" max="30" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-2 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Difficulty Walking/Climbing Stairs?</label>
                        <select id="DiffWalk" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">No</option>
                            <option value="1">Yes</option>
                        </select>
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Biological Sex</label>
                        <select id="Sex" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                            <option value="0">Female</option>
                            <option value="1">Male</option>
                        </select>
                    </div>
                </div>

                <div class="grid grid-cols-1 sm:grid-cols-3 gap-4">
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Age Category (1-13)</label>
                        <input type="number" id="Age" value="9" min="1" max="13" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Education (1-6)</label>
                        <input type="number" id="Education" value="5" min="1" max="6" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                    <div>
                        <label class="block text-sm font-medium text-gray-700">Income (1-8)</label>
                        <input type="number" id="Income" value="6" min="1" max="8" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm focus:border-indigo-500 focus:ring-indigo-500 border p-2">
                    </div>
                </div>

                <button type="button" onclick="runTinyMLInference()" class="w-full bg-indigo-600 hover:bg-indigo-700 text-white font-bold py-3 px-4 rounded-xl transition duration-150">
                    Run Local TinyML Prediction
                </button>
            </form>
        </section>

        <!-- Right Column: Interactive Results dashboard -->
        <section class="space-y-6">
            <!-- TinyML Card -->
            <div class="bg-white p-6 rounded-2xl shadow-sm border border-gray-100 flex flex-col justify-between">
                <h2 class="text-xl font-bold text-gray-800 mb-2">On-Device Risk Classification</h2>
                <div class="flex items-center space-x-6 mt-2">
                    <div class="relative w-28 h-28 flex items-center justify-center rounded-full bg-gray-100 border-4 border-indigo-200">
                        <span id="riskPercentage" class="text-2xl font-black text-indigo-600">--%</span>
                    </div>
                    <div>
                        <p class="text-sm font-semibold text-gray-500 uppercase tracking-wider">Classification Status</p>
                        <p id="riskClass" class="text-lg font-bold text-gray-400">Waiting for Inference...</p>
                        <p class="text-xs text-gray-400 mt-1">Calculated locally on-device without network latency</p>
                    </div>
                </div>
            </div>

            <!-- Agentic Explanation Card -->
            <div class="bg-white p-6 rounded-2xl shadow-sm border border-gray-100">
                <div class="flex justify-between items-center mb-4">
                    <h2 class="text-xl font-bold text-gray-800">Agentic Clinical Explanation</h2>
                    <button id="ragBtn" onclick="fetchRAGReport()" disabled class="bg-emerald-600 hover:bg-emerald-700 disabled:bg-gray-300 disabled:cursor-not-allowed text-white text-xs font-bold py-2 px-4 rounded-lg transition">
                        Generate RAG Report
                    </button>
                </div>
                <div class="bg-gray-50 p-4 rounded-xl border border-gray-100 min-h-[250px] max-h-[400px] overflow-y-auto">
                    <div id="ragReport" class="text-sm text-gray-500 whitespace-pre-wrap leading-relaxed">
                        Please run the local prediction first, then click "Generate RAG Report" to query Supabase and synthesize clinical advice.
                    </div>
                </div>
            </div>
        </section>
    </main>

    <script>
        let session = null;
        let lastPredictionResult = null;
        let lastFeatures = null;

        // Load the ONNX model as soon as the page opens
        async function loadModel() {
            try {
                // Initialize ONNX inference session pointing to local directory
                session = await ort.InferenceSession.create('./diabetes_predictor.onnx');
                console.log("ONNX Model loaded successfully in browser.");
            } catch (e) {
                console.error("Failed to load local ONNX model:", e);
                alert("Could not load the local diabetes_predictor.onnx model. Please make sure it is saved in the same directory as this index.html file.");
            }
        }

        async function runTinyMLInference() {
            if (!session) {
                alert("Model is still loading or could not be loaded.");
                return;
            }

            // Extract values strictly in the matching index order of training columns
            const orderOfFeatures = [
                'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 
                'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 
                'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 
                'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income'
            ];

            const floatArray = [];
            lastFeatures = {};

            orderOfFeatures.forEach(feat => {
                const value = parseFloat(document.getElementById(feat).value);
                floatArray.push(value);
                lastFeatures[feat] = value;
            });

            // Convert array into standard float32 Tensor shape [1, 21]
            const inputTensor = new ort.Tensor('float32', Float32Array.from(floatArray), [1, 21]);
            const feeds = { input: inputTensor };

            try {
                // Execute inference local inside browser
                const outputs = await session.run(feeds);
                const probabilities = outputs.probabilities.data; // Float32Array containing probabilities for class [0, 1]
                const riskProbability = probabilities[1]; // Index 1 is Class 1 (diabetic risk)

                lastPredictionResult = riskProbability;

                // Update UI dashboard
                document.getElementById('riskPercentage').innerText = (riskProbability * 100).toFixed(2) + "%";
                const riskClass = document.getElementById('riskClass');
                
                if (riskProbability >= 0.5) {
                    riskClass.innerText = "High Risk Level Identified";
                    riskClass.className = "text-lg font-bold text-red-600";
                } else {
                    riskClass.innerText = "Normal / Low Risk Level";
                    riskClass.className = "text-lg font-bold text-emerald-600";
                }

                // Enable the RAG button once prediction is calculated
                document.getElementById('ragBtn').disabled = false;
            } catch (err) {
                console.error("Inference failed:", err);
            }
        }

        async function fetchRAGReport() {
            if (lastPredictionResult === null || lastFeatures === null) return;

            const ragReportDiv = document.getElementById('ragReport');
            ragReportDiv.innerText = "Querying Supabase and running synthesis via Gemini AI...";

            try {
                // Send payload to our local Flask backend
                const response = await fetch('http://127.0.0.1:5000/api/explain', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({
                        risk_score: lastPredictionResult,
                        features: lastFeatures
                    })
                });

                const data = await response.json();
                if (data.report) {
                    ragReportDiv.innerText = data.report;
                    ragReportDiv.className = "text-sm text-gray-700 whitespace-pre-wrap leading-relaxed font-sans";
                } else {
                    ragReportDiv.innerText = "Error: Backend returned unexpected response.";
                }
            } catch (e) {
                console.error("RAG request failed:", e);
                ragReportDiv.innerText = "Failed to connect to local backend. Make sure server.py is running on port 5000.";
            }
        }

        window.onload = loadModel;
    </script>
</body>
</html>